# 3. Exploratory Visualization

**Summary**: Interactive exploration of MAL curves, heatmaps, and basic scatter plots for data validation and initial insights.

**Key Steps**:
1. Load analysis results.
2. Plot MAL curves (Mean Aggregate Length).
3. Generate heatmaps of dependency patterns.

**Inputs**:
- `data/all_langs_average_sizes.pkl`
- `data/lang2MAL.pkl`

**Outputs**:
- Inline plots for exploration.

**Runtime**: Fast (< 30s)

---

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import custom modules
import data_utils
import analysis
import plotting

%matplotlib inline

## Configuration

In [ ]:
DATA_DIR = "data"
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

## 1. Load Data

In [ ]:
# Load metadata
metadata = data_utils.load_metadata(os.path.join(DATA_DIR, 'metadata.pkl'))

langNames = metadata['langNames']
langnameGroup = metadata['langnameGroup']
appearance_dict = metadata['appearance_dict']

print(f"Languages: {len(langNames)}")
print(f"Groups: {len(set(langnameGroup.values()))}")

In [ ]:
# Load analysis results
results = analysis.load_analysis_results(DATA_DIR)

all_langs_position2num = results['all_langs_position2num']
all_langs_position2sizes = results['all_langs_position2sizes']
all_langs_average_sizes = results['all_langs_average_sizes']
filtered_position2num = results['filtered_position2num']
filtered_position2sizes = results['filtered_position2sizes']
lang2MAL = results['lang2MAL']

print(f"Languages with results: {len(lang2MAL)}")

## 2. Explore MAL Data

In [ ]:
# Show sample MAL data
sample_langs = list(lang2MAL.keys())[:5]
print("Sample MAL data:")
for lang in sample_langs:
    lang_name = langNames.get(lang, lang)
    mal_values = lang2MAL[lang]
    print(f"  {lang_name} ({lang}): {mal_values}")

In [ ]:
# Statistics on MAL coverage
max_n_per_lang = {lang: max(mal.keys()) for lang, mal in lang2MAL.items() if mal}
print(f"Maximum n value per language:")
print(pd.Series(max_n_per_lang).describe())
print(f"\nLanguages with MAL data: {len(max_n_per_lang)}")
print(f"Languages without MAL data: {len(lang2MAL) - len(max_n_per_lang)}")

## 3. Plot MAL Curves (n=2)

In [ ]:
plotting.plot_mal_curves_with_groups(
    lang2MAL,
    langNames,
    langnameGroup,
    appearance_dict,
    n_required=2,
    figsize=(12, 7)
)

## 4. Plot MAL Curves (n=3)

In [ ]:
plotting.plot_mal_curves_with_groups(
    lang2MAL,
    langNames,
    langnameGroup,
    appearance_dict,
    n_required=3,
    figsize=(12, 7)
)

## 5. Plot MAL Curves (n=5)

In [ ]:
plotting.plot_mal_curves_with_groups(
    lang2MAL,
    langNames,
    langnameGroup,
    appearance_dict,
    n_required=5,
    figsize=(12, 7)
)

## 6. Plot MAL Heatmap

In [ ]:
plotting.plot_mal_heatmap(
    lang2MAL,
    figsize=(12, 20),
    cmap="coolwarm"
)

## 7. Position Distribution Analysis

Explore the distribution of specific position types across languages.

In [ ]:
# Plot distribution of right_1 position
plotting.plot_position_distribution(
    all_langs_average_sizes,
    'right_1',
    langNames,
    langnameGroup,
    appearance_dict,
    top_n=30
)

In [ ]:
# Plot distribution of left_1 position
plotting.plot_position_distribution(
    all_langs_average_sizes,
    'left_1',
    langNames,
    langnameGroup,
    appearance_dict,
    top_n=30
)

## 8. Create DataFrame for 2D/3D Analysis

In [ ]:
# Create DataFrame with MAL values for n=1,2,3
df_mal = []
for lang, mal_dict in lang2MAL.items():
    if 1 in mal_dict and 2 in mal_dict and 3 in mal_dict:
        lang_name = langNames.get(lang, lang)
        group = langnameGroup.get(lang_name, 'Other')
        df_mal.append({
            'language': lang_name,
            'code': lang,
            'group': group,
            'MAL1': mal_dict[1],
            'MAL2': mal_dict[2],
            'MAL3': mal_dict[3]
        })

df_mal = pd.DataFrame(df_mal)
print(f"Languages with MAL1, MAL2, MAL3: {len(df_mal)}")
df_mal.head()

## 9. 2D Scatter Plot: MAL1 vs MAL2

In [ ]:
plotting.plot_scatter_2d(
    df_mal,
    'MAL1',
    'MAL2',
    'group',
    appearance_dict,
    title="MAL₂ vs MAL₁ across languages",
    xlabel="MAL₁ (1 right dependent)",
    ylabel="MAL₂ (2 right dependents)",
    figsize=(12, 9)
)

## 10. 3D Scatter Plot: MAL1 vs MAL2 vs MAL3

In [ ]:
plotting.plot_scatter_3d(
    df_mal,
    'MAL1',
    'MAL2',
    'MAL3',
    'group',
    appearance_dict,
    title="MAL across languages (3D)",
    xlabel="MAL₁",
    ylabel="MAL₂",
    zlabel="MAL₃",
    figsize=(14, 12)
)

## 11. Group Statistics

In [ ]:
# Compute group statistics
group_stats = df_mal.groupby('group')[['MAL1', 'MAL2', 'MAL3']].agg(['mean', 'std', 'count'])
print("Group statistics:")
print(group_stats)

In [ ]:
# Plot group means
group_means = df_mal.groupby('group')[['MAL1', 'MAL2', 'MAL3']].mean()
group_means.plot(kind='bar', figsize=(12, 6), rot=45)
plt.title("Mean MAL values by language group")
plt.xlabel("Language group")
plt.ylabel("MAL")
plt.legend(title="n dependents")
plt.tight_layout()

# Save plot
import os
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/group_means_mal.png', bbox_inches='tight', dpi=150)
print("✓ Saved: plots/group_means_mal.png")

plt.show()

## 12. Custom Analysis

Use this section for your own exploratory analysis.

In [ ]:
# Your custom analysis here

## Summary

This notebook has:
- ✅ Loaded analysis results from notebook 02
- ✅ Plotted MAL curves for n=2, 3, 5
- ✅ Created MAL heatmap across all languages
- ✅ Analyzed position distributions (left_1, right_1)
- ✅ Created 2D scatter plot (MAL1 vs MAL2)
- ✅ Created 3D scatter plot (MAL1 vs MAL2 vs MAL3)
- ✅ Computed group-level statistics

**Key observations**:
- MAL typically increases with n (more dependents → longer dependencies)
- Clear differences between language groups
- Position distributions vary significantly across languages